In [1]:
%pip install selenium pandas beautifulsoup4 ipython openpyxl

321.57s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

from bs4 import BeautifulSoup

import pandas as pd
from IPython.display import clear_output

from time import sleep
from datetime import datetime
from urllib.parse import quote

In [3]:
location = "Москва"
title = "салон красоты"

# Для первого запуска ставим всего 10
count_of_units = 10

print("Город:", location)
print("Поиск:", title)
print("Количество:", count_of_units)

Город: Москва
Поиск: салон красоты
Количество: 10


In [4]:
# Если старый driver существует — закрываем его
try:
    driver.quit()
except:
    pass

options = Options()

# Не отключаем картинки пока.
# Нам сейчас важнее стабильность, чем скорость.

driver = webdriver.Chrome(options=options)

print("Chrome запущен")

Chrome запущен


In [5]:
search_query = f"{location} {title}"

url = f"https://yandex.ru/maps/?text={quote(search_query)}"

print("Открываем:")
print(url)

driver.get(url)

sleep(5)

print("Название страницы:")
print(driver.title)

Открываем:
https://yandex.ru/maps/?text=%D0%9C%D0%BE%D1%81%D0%BA%D0%B2%D0%B0%20%D1%81%D0%B0%D0%BB%D0%BE%D0%BD%20%D0%BA%D1%80%D0%B0%D1%81%D0%BE%D1%82%D1%8B
Название страницы:
Яндекс Карты — транспорт, навигация, поиск мест


In [6]:
print("Открытых окон:", len(driver.window_handles))
print("URL:", driver.current_url)
print("Title:", driver.title)

Открытых окон: 1
URL: https://yandex.ru/maps/213/moscow/search/%D0%9C%D0%BE%D1%81%D0%BA%D0%B2%D0%B0%20%D1%81%D0%B0%D0%BB%D0%BE%D0%BD%20%D0%BA%D1%80%D0%B0%D1%81%D0%BE%D1%82%D1%8B/?ll=37.593029%2C55.762061&sll=37.593029%2C55.761908&z=10
Title: Яндекс Карты — транспорт, навигация, поиск мест


In [7]:
selectors = [
    "a[href*='/org/']",
    "a[href*='/maps/org/']"
]

elements = []

for selector in selectors:
    found = driver.find_elements(By.CSS_SELECTOR, selector)
    
    print(f"{selector}: найдено {len(found)}")
    
    if len(found) > len(elements):
        elements = found

print("Всего найдено элементов:", len(elements))

a[href*='/org/']: найдено 15
a[href*='/maps/org/']: найдено 15
Всего найдено элементов: 15


In [8]:
previous_count = 0
same_count = 0

while len(elements) < count_of_units and same_count < 10:
    
    # Прокручиваем страницу вниз
    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )
    
    sleep(2)
    
    # Снова ищем организации
    elements = []
    
    for selector in selectors:
        found = driver.find_elements(By.CSS_SELECTOR, selector)
        
        if len(found) > len(elements):
            elements = found
    
    current_count = len(elements)
    
    print("Найдено:", current_count)
    
    if current_count == previous_count:
        same_count += 1
    else:
        same_count = 0
    
    previous_count = current_count

print()
print("Финальное количество:", len(elements))


Финальное количество: 15


In [9]:
href_list = []

for element in elements:
    try:
        href = element.get_attribute("href")

        if href and "/org/" in href:
            href_list.append(href)

    except:
        pass


# Убираем дубликаты
href_list = list(dict.fromkeys(href_list))

# Ограничиваем количество
href_list = href_list[:count_of_units]

print("Уникальных организаций:", len(href_list))

for link in href_list:
    print(link)

Уникальных организаций: 10
https://yandex.ru/maps/org/studiya_modnykh_volos_pavla_okhapkina/1153600085/
https://yandex.ru/maps/org/studiya_modnykh_volos_pavla_okhapkina/1153600085/reviews/
https://yandex.ru/maps/org/studiya_modnykh_volos_pavla_okhapkina/1153600085/gallery/
https://yandex.ru/maps/org/wo_man/53550756563/
https://yandex.ru/maps/org/wo_man/53550756563/reviews/
https://yandex.ru/maps/org/wo_man/53550756563/gallery/
https://yandex.ru/maps/org/nails_up_taganskaya/45849278196/
https://yandex.ru/maps/org/nails_up_taganskaya/45849278196/reviews/
https://yandex.ru/maps/org/nails_up_taganskaya/45849278196/gallery/
https://yandex.ru/maps/org/stilno_i_tochka/202436168160/


In [10]:
data = {
    "href": [],
    "name": [],
    "address": [],
    "phone": [],
    "rating": [],
    "reviews": [],
    "site": []
}

print("Структура создана")

Структура создана


In [11]:
def clean_text(value):
    if value is None:
        return None
    
    return " ".join(value.split())

In [12]:
for number, link in enumerate(href_list, start=1):
    
    clear_output(wait=True)
    
    print(f"Обрабатываем {number}/{len(href_list)}")
    
    try:
        driver.get(link)
        
        # Даём странице загрузиться
        sleep(2)
        
        # Получаем HTML
        source = driver.page_source
        
        soup = BeautifulSoup(source, "html.parser")
        
        # -------------------------
        # Ссылка
        # -------------------------
        
        data["href"].append(link)
        
        
        # -------------------------
        # Название
        # -------------------------
        
        name = None
        
        h1 = soup.find("h1")
        
        if h1:
            name = clean_text(h1.get_text())
        
        data["name"].append(name)
        
        
        # -------------------------
        # Адрес
        # -------------------------
        
        address = None
        
        # Ищем элементы, содержащие address
        address_element = soup.select_one(
            "[class*='address']"
        )
        
        if address_element:
            address = clean_text(
                address_element.get_text()
            )
        
        data["address"].append(address)
        
        
        # -------------------------
        # Телефон
        # -------------------------
        
        phone = None
        
        phone_element = soup.select_one(
            "[class*='phone']"
        )
        
        if phone_element:
            phone = clean_text(
                phone_element.get_text()
            )
        
        data["phone"].append(phone)
        
        
        # -------------------------
        # Рейтинг
        # -------------------------
        
        rating = None
        
        rating_element = soup.select_one(
            "[class*='rating']"
        )
        
        if rating_element:
            rating = clean_text(
                rating_element.get_text()
            )
        
        data["rating"].append(rating)
        
        
        # -------------------------
        # Сайт
        # -------------------------
        
        site = None
        
        for a in soup.find_all("a", href=True):
            
            href = a.get("href", "")
            
            if (
                href.startswith("http")
                and "yandex.ru" not in href
                and "yandex.com" not in href
            ):
                site = href
                break
        
        data["site"].append(site)
        
        
        # -------------------------
        # Отзывы
        # -------------------------
        
        reviews = None
        
        # Ищем текстовые элементы,
        # связанные с отзывами
        for element in soup.find_all(string=True):
            
            text = clean_text(element)
            
            if text and "отзыв" in text.lower():
                reviews = text
                break
        
        data["reviews"].append(reviews)
        
        
    except Exception as error:
        
        print("Ошибка:", error)
        
        # Чтобы длина всех колонок оставалась одинаковой
        data["href"].append(link)
        data["name"].append(None)
        data["address"].append(None)
        data["phone"].append(None)
        data["rating"].append(None)
        data["reviews"].append(None)
        data["site"].append(None)
        
        sleep(1)

print("Готово!")

Обрабатываем 10/10
Готово!


In [13]:
df = pd.DataFrame(data)

df

,href,name,address,phone,rating,reviews,site
0,https://yandex.ru/maps/org/studiya_modnykh_vol...,Студия модных волос Павла Охапкина,"Краснопролетарская ул., 35, Москва",+7 (903) 546-51-61Показать телефон,"Студия Павла Охапкина4,9Стрижка: от 3300 ₽Поис...",window.document.head.insertAdjacentHTML('befor...,https://okhapkinstudio.ru/
1,https://yandex.ru/maps/org/studiya_modnykh_vol...,Студия модных волос Павла Охапкина,NaN,NaN,"Студия Павла Охапкина4,9Стрижка: от 3300 ₽Поис...",Отзывы о «Студия модных волос Павла Охапкина» ...,https://ya.ru
2,https://yandex.ru/maps/org/studiya_modnykh_vol...,Студия модных волос Павла Охапкина,NaN,NaN,"Студия Павла Охапкина4,9Стрижка: от 3300 ₽Поис...",Отзывы,https://ya.ru
3,https://yandex.ru/maps/org/wo_man/53550756563/,Wo/man,"ул. Преображенский Вал, 4, Москва",+7 (499) 444-16-04Показать телефон,"Wo/man4,9Стрижка: от 500 ₽ПоискМаршрутыКарты ·...",window.document.head.insertAdjacentHTML('befor...,https://womanstudios.ru/
4,https://yandex.ru/maps/org/wo_man/53550756563/...,Wo/man,NaN,NaN,"Wo/man4,9Стрижка: от 500 ₽ПоискМаршрутыWo/manО...","Отзывы о «Wo/man» на Преображенской площади, М...",https://ya.ru
5,https://yandex.ru/maps/org/wo_man/53550756563/...,Wo/man,NaN,NaN,"Wo/man4,9Стрижка: от 500 ₽ПоискМаршрутыWo/manО...",Отзывы,https://ya.ru
6,https://yandex.ru/maps/org/nails_up_taganskaya...,Nails Up Таганская,"Таганская площадь, 12, Москваслева от ресторан...",+7 (903) 713-13-54Показать телефон,"Nails Up Таганская4,8Закрыто до завтраПоискМар...",window.document.head.insertAdjacentHTML('befor...,https://nails-up-taganka.ru/
7,https://yandex.ru/maps/org/nails_up_taganskaya...,Nails Up Таганская,NaN,NaN,"Nails Up Таганская4,8Закрыто до завтраПоискМар...","Отзывы о «Nails Up Таганская» на Таганской, Мо...",https://ya.ru
8,https://yandex.ru/maps/org/nails_up_taganskaya...,Nails Up Таганская,NaN,NaN,"Nails Up Таганская4,8Закрыто до завтраПоискМар...",Отзывы,https://ya.ru
9,https://yandex.ru/maps/org/stilno_i_tochka/202...,Стильно и точка,"Ленинградский просп., 60, корп. 1, Москва",+7 (495) 120-42-40 (доб. 1)Показать телефон,"Стильно и точка4,9Стрижка: от 700 ₽ПоискМаршру...",window.document.head.insertAdjacentHTML('befor...,http://www.econom-studio.com/kosmetologiya/epi...


In [14]:
df[
    [
        "name",
        "address",
        "phone",
        "rating",
        "reviews",
        "site"
    ]
]

,name,address,phone,rating,reviews,site
0,Студия модных волос Павла Охапкина,"Краснопролетарская ул., 35, Москва",+7 (903) 546-51-61Показать телефон,"Студия Павла Охапкина4,9Стрижка: от 3300 ₽Поис...",window.document.head.insertAdjacentHTML('befor...,https://okhapkinstudio.ru/
1,Студия модных волос Павла Охапкина,NaN,NaN,"Студия Павла Охапкина4,9Стрижка: от 3300 ₽Поис...",Отзывы о «Студия модных волос Павла Охапкина» ...,https://ya.ru
2,Студия модных волос Павла Охапкина,NaN,NaN,"Студия Павла Охапкина4,9Стрижка: от 3300 ₽Поис...",Отзывы,https://ya.ru
3,Wo/man,"ул. Преображенский Вал, 4, Москва",+7 (499) 444-16-04Показать телефон,"Wo/man4,9Стрижка: от 500 ₽ПоискМаршрутыКарты ·...",window.document.head.insertAdjacentHTML('befor...,https://womanstudios.ru/
4,Wo/man,NaN,NaN,"Wo/man4,9Стрижка: от 500 ₽ПоискМаршрутыWo/manО...","Отзывы о «Wo/man» на Преображенской площади, М...",https://ya.ru
5,Wo/man,NaN,NaN,"Wo/man4,9Стрижка: от 500 ₽ПоискМаршрутыWo/manО...",Отзывы,https://ya.ru
6,Nails Up Таганская,"Таганская площадь, 12, Москваслева от ресторан...",+7 (903) 713-13-54Показать телефон,"Nails Up Таганская4,8Закрыто до завтраПоискМар...",window.document.head.insertAdjacentHTML('befor...,https://nails-up-taganka.ru/
7,Nails Up Таганская,NaN,NaN,"Nails Up Таганская4,8Закрыто до завтраПоискМар...","Отзывы о «Nails Up Таганская» на Таганской, Мо...",https://ya.ru
8,Nails Up Таганская,NaN,NaN,"Nails Up Таганская4,8Закрыто до завтраПоискМар...",Отзывы,https://ya.ru
9,Стильно и точка,"Ленинградский просп., 60, корп. 1, Москва",+7 (495) 120-42-40 (доб. 1)Показать телефон,"Стильно и точка4,9Стрижка: от 700 ₽ПоискМаршру...",window.document.head.insertAdjacentHTML('befor...,http://www.econom-studio.com/kosmetologiya/epi...


In [15]:
now = datetime.now()

date_time_str = now.strftime("%Y-%m-%d_%H-%M-%S")

filename = f"{location}_{title}_{date_time_str}.xlsx"

df.to_excel(filename, index=False)

print(f"Файл сохранён: {filename}")

Файл сохранён: Москва_салон красоты_2026-08-26_23-36-41.xlsx


In [16]:
driver.quit()

print("Chrome закрыт")

Chrome закрыт
